# Relational Deep Learning (RDL) Pipeline

Relational Learning on Relational Databases: Deep learning across multi-table relational databases via heterogeneous graph modeling. This notebook implements the approach with `HeteroConv` inside a `K3HeteroSAGE` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `HeteroConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Relational Deep Learning (RDL) Pipeline with Heterogeneous GraphSAGE"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Heterogeneous GraphSAGE Block
class K3HeteroSAGE(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3HeteroSAGE(in_channels=32, hidden_channels=64, out_channels=1)

# 2. Forward pass test
num_nodes = 50
dummy_x = keras.random.normal((num_nodes, 32))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_x, dummy_edges)
print(f"RDL node regression prediction shape: {out.shape}")

print("\n✓ K3-Node RDL execution completed successfully!")